### Extracting Trope informations from TEI XML files and adding to metadata files
after this need to add reprint information

In [1]:
import os
import pandas as pd
import xml.etree.ElementTree as ET

In [2]:
# ---------- CONFIG ----------
METADATA_PATH = "images_with_article_metadata.csv"  # <-- your current metadata sheet
OUTPUT_PATH = "metadata_with_tropes.csv"
OBJECTS_DIR = "objects"         # directory that contains the article folders

# mapping from n="…" to trope label
TROPE_MAP = {
    "1": "Indian Queen",
    "2": "Captivity",
    "3": "Noble Savage",
    "4": "Discovery",
    "5": "Wonder",
    "6": "Vanishing Indians",
    "7": "Origins",
    "8": "Girl Crusoe",
    "9": "Jump Overboard",
    "10": "Signs Easily Interpreted",
    "11": "Indians as Savage",
    "12": "No Boat Available",
    "13": "Anti-Russian Sentiment",
    "14": "Wild Child",
    "15": "Lazy Indian",
}

# ---------- FUNCTIONS ----------

def extract_tropes_from_xml(xml_path):
    """
    Given a path to a TEI XML file, return a semicolon-separated string of trope labels.
    - Looks for <span type="trope" n="X">
    - Maps X via TROPE_MAP
    - Deduplicates and returns labels in first-seen order
    """
    if not os.path.exists(xml_path):
        return ""

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        print(f"WARNING: Could not parse XML: {xml_path}")
        return ""

    labels_in_order = []

    # Iterate over all elements; handle namespaces by just using .endswith("span")
    for el in root.iter():
        if not el.tag.endswith("span"):
            continue
        if el.get("type") != "trope":
            continue

        n_val = el.get("n")
        if n_val is None:
            continue

        label = TROPE_MAP.get(n_val)
        if label is None:
            # Unknown trope number; you can log or ignore
            print(f"WARNING: Unknown trope number n='{n_val}' in {xml_path}")
            continue

        # Deduplicate while preserving order
        if label not in labels_in_order:
            labels_in_order.append(label)

    if not labels_in_order:
        return ""

    return "; ".join(labels_in_order)


# ---------- MAIN WORKFLOW ----------

def main():
    # 1. Load metadata
    starter_metadata_df = pd.read_csv(METADATA_PATH)

    if "article_id" not in starter_metadata_df.columns:
        raise ValueError("metadata CSV must contain an 'article_id' column")

    # Ensure 'tropes' column exists
    if "tropes" not in starter_metadata_df.columns:
        starter_metadata_df["tropes"] = ""

    # 2. For each unique article_id, extract tropes from the TEI file
    article_ids = starter_metadata_df["article_id"].dropna().unique()

    trope_by_article = {}

    for article_id in article_ids:
        # article_id might be float/number; make sure it's a string
        article_id_str = str(article_id)

        # Path like: objects/AlbanyEveningJournal1853/m_AlbanyEveningJournal1853_TEI.xml
        xml_path = os.path.join(
            OBJECTS_DIR,
            article_id_str,
            f"m_{article_id_str}_TEI.xml"
        )

        tropes = extract_tropes_from_xml(xml_path)
        trope_by_article[article_id_str] = tropes

    # 3. Fill the 'tropes' column for each row based on article_id
    def lookup_tropes(row):
        article_id_str = str(row["article_id"])
        return trope_by_article.get(article_id_str, "")

    starter_metadata_df["tropes"] = starter_metadata_df.apply(lookup_tropes, axis=1)

    # 4. Save output
    starter_metadata_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved updated metadata with tropes to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


Saved updated metadata with tropes to: metadata_with_tropes.csv
